In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import trimesh
import pyrender

# ===== app.py content =====
# Load mesh once c
MESH_PATH = "PureCadence_2_Color_HiRskRedNghtlfeSlvrBlckWht_Size_70/meshes/model.obj"  # Update path as needed
shoe_trimesh = trimesh.load(MESH_PATH, force='mesh')

# Scale if needed (e.g., mm to meters)
# shoe_trimesh.apply_scale(0.001)

# Create mesh and light
shoe_mesh = pyrender.Mesh.from_trimesh(shoe_trimesh, smooth=False)
light = pyrender.DirectionalLight(color=np.ones(3), intensity=3.0)

def make_base_scene(fx, fy, cx, cy):
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2])
    cam = pyrender.IntrinsicsCamera(fx, fy, cx, cy)
    cam_node = scene.add(cam, pose=np.eye(4))
    light_pose = np.array([[1, 0, 0, 0],
                          [0, 1, 0, 0],
                          [0, 0, 1, 4],
                          [0, 0, 0, 1]], dtype=np.float32)
    light_node = scene.add(light, pose=light_pose)
    shoe_node = scene.add(shoe_mesh, pose=np.eye(4))
    return scene, shoe_node

def alpha_composite(background_bgr, render_rgba):
    rgb = render_rgba[..., :3]
    a = render_rgba[..., 3:4] / 255.0
    rgb_bgr = rgb[..., ::-1]
    out = background_bgr.astype(np.float32) * (1 - a) + rgb_bgr.astype(np.float32) * a
    return out.astype(np.uint8)

# OpenCV to OpenGL coordinate system transformation
T_CV2GL = np.array([[1, 0, 0, 0],
                    [0, -1, 0, 0],
                    [0, 0, -1, 0],
                    [0, 0, 0, 1]], dtype=np.float32)

def render_on_frame(frame_bgr, foot_transform_cv, fx, fy, cx, cy, renderer_cache={}):
    H, W = frame_bgr.shape[:2]
    key = (W, H, fx, fy, cx, cy)
    
    if key not in renderer_cache:
        scene, shoe_node = make_base_scene(fx, fy, cx, cy)
        renderer = pyrender.OffscreenRenderer(viewport_width=W, viewport_height=H)
        renderer_cache[key] = (scene, shoe_node, renderer)
    else:
        scene, shoe_node, renderer = renderer_cache[key]
    
    # Apply coordinate system transformation
    pose_gl = T_CV2GL @ foot_transform_cv
    scene.set_pose(shoe_node, pose=pose_gl)
    
    color_rgba, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    return alpha_composite(frame_bgr, color_rgba)

In [5]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    model_complexity=1,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# Camera intrinsics
fx, fy = 900.0, 900.0
cx, cy = 320.0, 240.0

# Open video
cap = cv2.VideoCapture('5.mp4')

# Output video writer
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_fixed1.mp4', fourcc, fps, (frame_width, frame_height))

renderer_cache = {}

def get_3d_foot_transform(landmarks, indices, w, h):
    """Calculate foot transformation matrix from landmarks"""
    
    # Get 2D landmark positions
    heel = landmarks[indices['heel']]
    toe = landmarks[indices['toe']]
    ankle = landmarks[indices['ankle']]
    
    # Check visibility
    if heel.visibility < 0.5 or toe.visibility < 0.5 or ankle.visibility < 0.5:
        return None
    
    # Convert to pixel coordinates
    heel_px = np.array([heel.x * w, heel.y * h])
    toe_px = np.array([toe.x * w, toe.y * h])
    ankle_px = np.array([ankle.x * w, ankle.y * h])
    
    # Calculate a 4th point for stable PnP solving
    # Create a virtual point to the side of the foot
    foot_vector = toe_px - heel_px
    foot_length = np.linalg.norm(foot_vector)
    foot_norm = foot_vector / foot_length
    
    # Perpendicular vector (to the right of the foot)
    perp_vector = np.array([-foot_norm[1], foot_norm[0]])
    
    # Side point at heel level, offset to the side
    side_px = heel_px + perp_vector * (foot_length * 0.3)
    
    # Create 3D reference points for the foot
    typical_foot_length_m = 0.25  # 25cm typical foot
    typical_foot_width_m = 0.1   # 10cm typical width
    
    foot_3d_points = np.array([
        [0, 0, 0],                                    # Heel
        [typical_foot_length_m, 0, 0],                # Toe  
        [0, 0, -typical_foot_length_m * 0.2],         # Ankle (above heel)
        [0, typical_foot_width_m * 0.5, 0],           # Side point
    ], dtype=np.float32)
    
    # 2D points
    pts_2d = np.array([heel_px, toe_px, ankle_px, side_px], dtype=np.float32)
    
    # Solve PnP with 4 points
    camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    success, rvec, tvec = cv2.solvePnP(
        foot_3d_points,
        pts_2d,
        cameraMatrix=camera_matrix,
        distCoeffs=None,
        flags=cv2.SOLVEPNP_SQPNP  # SQPNP works well with exactly 4 points
    )
    
    if not success:
        return None
    
    # Convert to transformation matrix
    R, _ = cv2.Rodrigues(rvec)
    transform = np.eye(4)
    transform[:3, :3] = R
    transform[:3, 3] = tvec.flatten()
    
    # Additional rotation adjustments for shoe orientation
    # This depends on how your 3D shoe model is oriented
    # You may need to adjust these values based on your shoe model
    
    # Example: Rotate 180 degrees around Y-axis if shoe is facing wrong direction
    # flip_y = np.array([[-1, 0, 0, 0],
    #                    [0, 1, 0, 0],
    #                    [0, 0, -1, 0],
    #                    [0, 0, 0, 1]], dtype=np.float32)
    # transform = transform @ flip_y
    
    # Example: Rotate 90 degrees around X-axis if shoe is pointing up
    # rotate_x_90 = np.array([[1, 0, 0, 0],
    #                         [0, 0, -1, 0],
    #                         [0, 1, 0, 0],
    #                         [0, 0, 0, 1]], dtype=np.float32)
    # transform = transform @ rotate_x_90
    
    return transform

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    h, w = frame.shape[:2]
    output_frame = frame.copy()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        
        for side in ['left', 'right']:
            if side == 'left':
                indices = {
                    'heel': mp_pose.PoseLandmark.LEFT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.LEFT_ANKLE.value
                }
            else:
                indices = {
                    'heel': mp_pose.PoseLandmark.RIGHT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.RIGHT_ANKLE.value
                }
            
            # Calculate foot transformation
            transform = get_3d_foot_transform(lm, indices, w, h)
            
            if transform is not None:
                # Render shoe
                shoe_frame = render_on_frame(
                    frame_bgr=output_frame.copy(),
                    foot_transform_cv=transform,
                    fx=fx, fy=fy, cx=cx, cy=cy,
                    renderer_cache=renderer_cache
                )
                
                # Blend shoe onto output
                output_frame = cv2.addWeighted(output_frame, 0.3, shoe_frame, 0.7, 0)
                
                # Debug: Draw landmarks
                for key, idx in indices.items():
                    pt = lm[idx]
                    if pt.visibility > 0.5:
                        x, y = int(pt.x * w), int(pt.y * h)
                        color = (0, 255, 0) if side == 'left' else (0, 0, 255)
                        cv2.circle(output_frame, (x, y), 5, color, -1)
                        cv2.putText(output_frame, key[:1], (x+5, y-5), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Save to output
    out.write(output_frame)
    
    # Progress indicator
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

# Cleanup
cap.release()
out.release()
pose.close()
print(f"Processing complete! Total frames: {frame_count}")


I0000 00:00:1753248778.068220   38172 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1753248778.204574   69294 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753248778.222468   69294 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processing complete! Total frames: 425


### trying final approach

In [20]:
import cv2
import numpy as np
import mediapipe as mp
import trimesh
import pyrender

# ===== app.py content =====
# Load mesh once
MESH_PATH = "PureCadence_2_Color_HiRskRedNghtlfeSlvrBlckWht_Size_70/meshes/model.obj"  # Update path as needed
shoe_trimesh = trimesh.load(MESH_PATH, force='mesh')

# Scale if needed (e.g., mm to meters)
# shoe_trimesh.apply_scale(0.001)

def make_base_scene(fx, fy, cx, cy):
    # Create fresh mesh instance for each scene to avoid context binding issues
    shoe_mesh = pyrender.Mesh.from_trimesh(shoe_trimesh, smooth=False)
    light = pyrender.DirectionalLight(color=np.ones(3), intensity=3.0)
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2])
    cam = pyrender.IntrinsicsCamera(fx, fy, cx, cy)
    cam_node = scene.add(cam, pose=np.eye(4))
    light_pose = np.array([[1, 0, 0, 0],
                          [0, 1, 0, 0],
                          [0, 0, 1, 4],
                          [0, 0, 0, 1]], dtype=np.float32)
    light_node = scene.add(light, pose=light_pose)
    shoe_node = scene.add(shoe_mesh, pose=np.eye(4))
    return scene, shoe_node

def alpha_composite(background_bgr, render_rgba):
    rgb = render_rgba[..., :3]
    a = render_rgba[..., 3:4] / 255.0
    rgb_bgr = rgb[..., ::-1]
    out = background_bgr.astype(np.float32) * (1 - a) + rgb_bgr.astype(np.float32) * a
    return out.astype(np.uint8)

# OpenCV to OpenGL coordinate system transformation
T_CV2GL = np.array([[1, 0, 0, 0],
                    [0, -1, 0, 0],
                    [0, 0, -1, 0],
                    [0, 0, 0, 1]], dtype=np.float32)

def render_on_frame(frame_bgr, foot_transform_cv, fx, fy, cx, cy, renderer_cache={}):
    H, W = frame_bgr.shape[:2]
    key = (W, H, fx, fy, cx, cy)
    
    if key not in renderer_cache:
        # Clean up any existing renderer to avoid conflicts
        for k, (scene, shoe_node, renderer) in list(renderer_cache.items()):
            if k != key:
                renderer.delete()
                del renderer_cache[k]
        
        # Create new scene and renderer
        scene, shoe_node = make_base_scene(fx, fy, cx, cy)
        renderer = pyrender.OffscreenRenderer(viewport_width=W, viewport_height=H)
        renderer_cache[key] = (scene, shoe_node, renderer)
    else:
        scene, shoe_node, renderer = renderer_cache[key]
    
    # Apply coordinate system transformation
    pose_gl = T_CV2GL @ foot_transform_cv
    scene.set_pose(shoe_node, pose=pose_gl)
    
    color_rgba, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    return alpha_composite(frame_bgr, color_rgba)

In [14]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    model_complexity=1,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# Camera intrinsics
fx, fy = 900.0, 900.0
cx, cy = 320.0, 240.0

# Open video
cap = cv2.VideoCapture('3.mp4')

# Output video writer
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_fixed123234.mp4', fourcc, fps, (frame_width, frame_height))

renderer_cache = {}

def get_3d_foot_transform(landmarks, indices, w, h):
    """Calculate foot transformation matrix from landmarks"""
    
    # Get 2D landmark positions
    heel = landmarks[indices['heel']]
    toe = landmarks[indices['toe']]
    ankle = landmarks[indices['ankle']]
    
    # Check visibility
    if heel.visibility < 0.5 or toe.visibility < 0.5 or ankle.visibility < 0.5:
        return None
    
    # Convert to pixel coordinates
    heel_px = np.array([heel.x * w, heel.y * h])
    toe_px = np.array([toe.x * w, toe.y * h])
    ankle_px = np.array([ankle.x * w, ankle.y * h])
    
    # Calculate foot direction in 2D
    foot_vector = toe_px - heel_px
    foot_length = np.linalg.norm(foot_vector)
    
    # Calculate the angle of the foot in the image
    foot_angle = np.arctan2(foot_vector[1], foot_vector[0])
    
    # Estimate depth based on foot size
    typical_foot_length_px = 150  # Expected foot length in pixels at reference distance
    scale_factor = typical_foot_length_px / foot_length
    z_distance = 2.0 * scale_factor  # Rough depth estimate
    
    # Calculate position (center of foot)
    foot_center_px = (heel_px + toe_px) / 2
    
    # Convert pixel coordinates to camera coordinates
    tvec = np.array([
        (foot_center_px[0] - cx) * z_distance / fx,
        (foot_center_px[1] - cy) * z_distance / fy,
        z_distance
    ])
    
    # Create rotation matrix
    # First, align shoe with foot direction
    transform = np.eye(4)
    
    # Rotation around Z-axis to align with foot angle
    cos_a = np.cos(foot_angle)
    sin_a = np.sin(foot_angle)
    
    # Basic rotation matrix for foot alignment
    R_align = np.array([
        [cos_a, -sin_a, 0],
        [sin_a, cos_a, 0],
        [0, 0, 1]
    ])
    
    # Apply rotations to properly orient the shoe
    # 1. First align with foot direction
    transform[:3, :3] = R_align
    
    # 2. Rotate to lay flat (assuming shoe model is vertical initially)
    # Rotate -90 degrees around the aligned X-axis
    rotate_x = np.array([
        [1, 0, 0],
        [0, 0, 1],
        [0, -1, 0]
    ])
    transform[:3, :3] = transform[:3, :3] @ rotate_x
    
    # 3. Additional 90-degree rotation around Z to properly orient the shoe
    rotate_z_90 = np.array([
        [0, -1, 0],
        [1, 0, 0],
        [0, 0, 1]
    ])
    transform[:3, :3] = transform[:3, :3] @ rotate_z_90
    
    # Set translation
    transform[:3, 3] = tvec
    
    # Fine-tune: Adjust vertical position to place shoe on foot
    transform[2, 3] -= 0.05  # Move shoe down slightly
    
    return transform

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    h, w = frame.shape[:2]
    output_frame = frame.copy()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        
        for side in ['left', 'right']:
            if side == 'left':
                indices = {
                    'heel': mp_pose.PoseLandmark.LEFT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.LEFT_ANKLE.value
                }
            else:
                indices = {
                    'heel': mp_pose.PoseLandmark.RIGHT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.RIGHT_ANKLE.value
                }
            
            # Calculate foot transformation
            transform = get_3d_foot_transform(lm, indices, w, h)
            
            if transform is not None:
                # Adjust position to center shoe on foot
                # Move shoe forward/backward based on your shoe model's origin
                # transform[0, 3] += 0.1  # Adjust X if needed
                # transform[1, 3] += 0.05  # Adjust Y if needed
                
                # Render shoe
                shoe_frame = render_on_frame(
                    frame_bgr=output_frame.copy(),
                    foot_transform_cv=transform,
                    fx=fx, fy=fy, cx=cx, cy=cy,
                    renderer_cache=renderer_cache
                )
                
                # Blend shoe onto output
                output_frame = cv2.addWeighted(output_frame, 0.2, shoe_frame, 0.8, 0)
                
                # Debug: Draw landmarks and foot direction
                for key, idx in indices.items():
                    pt = lm[idx]
                    if pt.visibility > 0.5:
                        x, y = int(pt.x * w), int(pt.y * h)
                        color = (0, 255, 0) if side == 'left' else (0, 0, 255)
                        cv2.circle(output_frame, (x, y), 5, color, -1)
                        cv2.putText(output_frame, key[:1], (x+5, y-5), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
                
                # Draw line from heel to toe to visualize foot direction
                heel_pt = lm[indices['heel']]
                toe_pt = lm[indices['toe']]
                if heel_pt.visibility > 0.5 and toe_pt.visibility > 0.5:
                    heel_xy = (int(heel_pt.x * w), int(heel_pt.y * h))
                    toe_xy = (int(toe_pt.x * w), int(toe_pt.y * h))
                    cv2.line(output_frame, heel_xy, toe_xy, (255, 255, 0), 2)
    
    # Save to output
    out.write(output_frame)
    
    # Progress indicator
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

# Cleanup
cap.release()
out.release()
pose.close()
print(f"Processing complete! Total frames: {frame_count}")

I0000 00:00:1753257844.216507   38172 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1753257844.319158  153504 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753257844.341997  153506 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processing complete! Total frames: 432


# efgdfghdfhg

In [1]:
# Alternative: Load with resolver for textures
import os
import cv2
import numpy as np
import mediapipe as mp
import trimesh
import pyrender

# ===== app.py content =====
# Load mesh once
MESH_PATH = "PureCadence_2_Color_HiRskRedNghtlfeSlvrBlckWht_Size_70/meshes/model.obj"  # Update path as needed
shoe_trimesh = trimesh.load(MESH_PATH, force='mesh')
# Load mesh with texture resolver
mesh_dir = os.path.dirname(MESH_PATH)
resolver = trimesh.visual.resolvers.FilePathResolver(mesh_dir)
shoe_trimesh = trimesh.load(MESH_PATH, force='mesh', process=True, 
                           visual=trimesh.visual.texture.TextureVisuals,
                           resolver=resolver)

# Debug: Check if textures loaded
print(f"Mesh loaded. Has visual: {hasattr(shoe_trimesh, 'visual')}")
if hasattr(shoe_trimesh, 'visual'):
    print(f"Visual type: {type(shoe_trimesh.visual)}")
    if hasattr(shoe_trimesh.visual, 'material'):
        print(f"Has material: True")
        print(f"Material: {shoe_trimesh.visual.material}")

# If the mesh has texture/material files (MTL), make sure they're in the same directory
# The OBJ loader should automatically load the MTL file if it exists

# Scale if needed (e.g., mm to meters)
# shoe_trimesh.apply_scale(0.001)

def make_base_scene(fx, fy, cx, cy):
    # Create mesh with materials/textures
    if hasattr(shoe_trimesh, 'visual') and hasattr(shoe_trimesh.visual, 'material'):
        # Mesh has materials/textures
        shoe_mesh = pyrender.Mesh.from_trimesh(shoe_trimesh, smooth=False)
    else:
        # Fallback: create mesh with a default colorful material
        # You can adjust these colors to match your shoe
        material = pyrender.MetallicRoughnessMaterial(
            baseColorFactor=[0.8, 0.3, 0.3, 1.0],  # RGBA color (reddish)
            metallicFactor=0.2,
            roughnessFactor=0.8
        )
        primitive = pyrender.Primitive(
            positions=shoe_trimesh.vertices,
            normals=shoe_trimesh.vertex_normals,
            indices=shoe_trimesh.faces,
            material=material
        )
        shoe_mesh = pyrender.Mesh(primitives=[primitive])
    
    # Add multiple lights for better illumination
    light1 = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=3.0)
    light2 = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=2.0)
    
    scene = pyrender.Scene(ambient_light=[0.3, 0.3, 0.3])  # Increased ambient light
    cam = pyrender.IntrinsicsCamera(fx, fy, cx, cy)
    cam_node = scene.add(cam, pose=np.eye(4))
    
    # Add lights from different angles
    light_pose1 = np.array([[1, 0, 0, 0],
                           [0, 1, 0, 0],
                           [0, 0, 1, 4],
                           [0, 0, 0, 1]], dtype=np.float32)
    light_pose2 = np.array([[0.7, 0, 0.7, 0],
                           [0, 1, 0, 0],
                           [-0.7, 0, 0.7, 4],
                           [0, 0, 0, 1]], dtype=np.float32)
    
    light_node1 = scene.add(light1, pose=light_pose1)
    light_node2 = scene.add(light2, pose=light_pose2)
    
    shoe_node = scene.add(shoe_mesh, pose=np.eye(4))
    return scene, shoe_node

def alpha_composite(background_bgr, render_rgba):
    rgb = render_rgba[..., :3]
    a = render_rgba[..., 3:4] / 255.0
    rgb_bgr = rgb[..., ::-1]
    out = background_bgr.astype(np.float32) * (1 - a) + rgb_bgr.astype(np.float32) * a
    return out.astype(np.uint8)

# OpenCV to OpenGL coordinate system transformation
T_CV2GL = np.array([[1, 0, 0, 0],
                    [0, -1, 0, 0],
                    [0, 0, -1, 0],
                    [0, 0, 0, 1]], dtype=np.float32)

def render_on_frame(frame_bgr, foot_transform_cv, fx, fy, cx, cy, renderer_cache={}):
    H, W = frame_bgr.shape[:2]
    key = (W, H, fx, fy, cx, cy)
    
    if key not in renderer_cache:
        # Clean up any existing renderer to avoid conflicts
        for k, (scene, shoe_node, renderer) in list(renderer_cache.items()):
            if k != key:
                renderer.delete()
                del renderer_cache[k]
        
        # Create new scene and renderer
        scene, shoe_node = make_base_scene(fx, fy, cx, cy)
        renderer = pyrender.OffscreenRenderer(viewport_width=W, viewport_height=H)
        renderer_cache[key] = (scene, shoe_node, renderer)
    else:
        scene, shoe_node, renderer = renderer_cache[key]
    
    # Apply coordinate system transformation
    pose_gl = T_CV2GL @ foot_transform_cv
    scene.set_pose(shoe_node, pose=pose_gl)
    
    color_rgba, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    return alpha_composite(frame_bgr, color_rgba)


Mesh loaded. Has visual: True
Visual type: <class 'trimesh.visual.texture.TextureVisuals'>
Has material: True
Material: <trimesh.visual.material.SimpleMaterial object at 0x179cc7dd0>


In [ ]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    model_complexity=1,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

# Camera intrinsics
fx, fy = 900.0, 900.0
cx, cy = 320.0, 240.0

# Open video
cap = cv2.VideoCapture('8.mp4')

# Output video writer
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_fixed5555.mp4', fourcc, fps, (frame_width, frame_height))

renderer_cache = {}

def get_3d_foot_transform(landmarks, indices, w, h):
    """Calculate foot transformation matrix from landmarks"""
    
    # Get 2D landmark positions
    heel = landmarks[indices['heel']]
    toe = landmarks[indices['toe']]
    ankle = landmarks[indices['ankle']]
    
    # Check visibility
    if heel.visibility < 0.5 or toe.visibility < 0.5 or ankle.visibility < 0.5:
        return None
    
    # Convert to pixel coordinates
    heel_px = np.array([heel.x * w, heel.y * h])
    toe_px = np.array([toe.x * w, toe.y * h])
    ankle_px = np.array([ankle.x * w, ankle.y * h])
    
    # Calculate foot direction in 2D
    foot_vector = toe_px - heel_px
    foot_length = np.linalg.norm(foot_vector)
    
    # Calculate the angle of the foot in the image
    foot_angle = np.arctan2(foot_vector[1], foot_vector[0])
    
    # Estimate depth based on foot size
    typical_foot_length_px = 150  # Expected foot length in pixels at reference distance
    scale_factor = typical_foot_length_px / foot_length
    z_distance = 2.0 * scale_factor  # Rough depth estimate
    
    # Calculate position (center of foot)
    foot_center_px = (heel_px + toe_px) / 2
    
    # Convert pixel coordinates to camera coordinates
    tvec = np.array([
        (foot_center_px[0] - cx) * z_distance / fx,
        (foot_center_px[1] - cy) * z_distance / fy,
        z_distance
    ])
    
    # Create rotation matrix
    # First, align shoe with foot direction
    transform = np.eye(4)
    
    # Rotation around Z-axis to align with foot angle
    cos_a = np.cos(foot_angle)
    sin_a = np.sin(foot_angle)
    
    # Basic rotation matrix for foot alignment
    R_align = np.array([
        [cos_a, -sin_a, 0],
        [sin_a, cos_a, 0],
        [0, 0, 1]
    ])
    
    # Apply rotations to properly orient the shoe
    # 1. First align with foot direction
    transform[:3, :3] = R_align
    
    # 2. Rotate to lay flat (assuming shoe model is vertical initially)
    # Rotate -90 degrees around the aligned X-axis
    rotate_x = np.array([
        [1, 0, 0],
        [0, 0, 1],
        [0, -1, 0]
    ])
    transform[:3, :3] = transform[:3, :3] @ rotate_x
    
    # 3. Additional 90-degree rotation around Z to properly orient the shoe
    rotate_z_90 = np.array([
        [0, -1, 0],
        [1, 0, 0],
        [0, 0, 1]
    ])
    transform[:3, :3] = transform[:3, :3] @ rotate_z_90
    
    # Set translation
    transform[:3, 3] = tvec
    
    # Fine-tune: Adjust vertical position to place shoe on foot
    transform[2, 3] -= 0.05  # Move shoe down slightly
    
    return transform

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    h, w = frame.shape[:2]
    output_frame = frame.copy()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)
    
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        
        for side in ['left', 'right']:
            if side == 'left':
                indices = {
                    'heel': mp_pose.PoseLandmark.LEFT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.LEFT_ANKLE.value
                }
            else:
                indices = {
                    'heel': mp_pose.PoseLandmark.RIGHT_HEEL.value,
                    'toe': mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value,
                    'ankle': mp_pose.PoseLandmark.RIGHT_ANKLE.value
                }
            
            # Calculate foot transformation
            transform = get_3d_foot_transform(lm, indices, w, h)
            
            if transform is not None:
                # Adjust position to center shoe on foot
                # Move shoe forward/backward based on your shoe model's origin
                # transform[0, 3] += 0.1  # Adjust X if needed
                # transform[1, 3] += 0.05  # Adjust Y if needed
                
                # Render shoe
                shoe_frame = render_on_frame(
                    frame_bgr=output_frame.copy(),
                    foot_transform_cv=transform,
                    fx=fx, fy=fy, cx=cx, cy=cy,
                    renderer_cache=renderer_cache
                )
                
                # Blend shoe onto output
                output_frame = cv2.addWeighted(output_frame, 0.2, shoe_frame, 0.8, 0)
                
                # Debug: Draw landmarks and foot direction
                for key, idx in indices.items():
                    pt = lm[idx]
                    if pt.visibility > 0.5:
                        x, y = int(pt.x * w), int(pt.y * h)
                        color = (0, 255, 0) if side == 'left' else (0, 0, 255)
                        cv2.circle(output_frame, (x, y), 5, color, -1)
                        cv2.putText(output_frame, key[:1], (x+5, y-5), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
                
                # Draw line from heel to toe to visualize foot direction
                heel_pt = lm[indices['heel']]
                toe_pt = lm[indices['toe']]
                if heel_pt.visibility > 0.5 and toe_pt.visibility > 0.5:
                    heel_xy = (int(heel_pt.x * w), int(heel_pt.y * h))
                    toe_xy = (int(toe_pt.x * w), int(toe_pt.y * h))
                    cv2.line(output_frame, heel_xy, toe_xy, (255, 255, 0), 2)
    
    # Save to output
    out.write(output_frame)
    
    # Progress indicator
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

# Cleanup
cap.release()
out.release()
pose.close()
print(f"Processing complete! Total frames: {frame_count}")


I0000 00:00:1753957949.543752  267888 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1753957949.626718  273603 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753957949.645981  273605 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753957949.662943  273604 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
2025-07-31 16:02:29.802 Python[6912:267888] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/50/f1bll6vs2cjgtlj7q9qcmrxw0000gn/T/org.python.python.savedState


Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processing complete! Total frames: 331


: 

In [ ]:
import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, enable_segmentation=False)
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = pose.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    if results.pose_landmarks:
        h, w, _ = image.shape
        landmarks = results.pose_landmarks.landmark
        foot_landmarks = {
            'LEFT_ANKLE': mp_pose.PoseLandmark.LEFT_ANKLE,
            'RIGHT_ANKLE': mp_pose.PoseLandmark.RIGHT_ANKLE,
            'LEFT_HEEL': mp_pose.PoseLandmark.LEFT_HEEL,
            'RIGHT_HEEL': mp_pose.PoseLandmark.RIGHT_HEEL,
            'LEFT_FOOT_INDEX': mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
            'RIGHT_FOOT_INDEX': mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
        }

        for name, idx in foot_landmarks.items():
            lm = landmarks[idx]
            x_2d, y_2d = int(lm.x * w), int(lm.y * h)
            z_3d = lm.z

            print(f"{name} - 2D: ({x_2d}, {y_2d}), 3D: ({lm.x:.4f}, {lm.y:.4f}, {lm.z:.4f})")
            cv2.circle(image, (x_2d, y_2d), 6, (0, 255, 0), -1)

        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    cv2.imshow('MediaPipe Feet Tracker', image)

    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


I0000 00:00:1753421520.933332   20455 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1753421521.017672   83803 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753421521.034335   83809 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2025-07-25 11:02:01.562 Python[2361:20455] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/50/f1bll6vs2cjgtlj7q9qcmrxw0000gn/T/org.python.python.savedState
W0000 00:00:1753421522.390603   83805 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


LEFT_ANKLE - 2D: (441, 580), 3D: (0.3447, 0.8067, 0.0205)
RIGHT_ANKLE - 2D: (400, 581), 3D: (0.3127, 0.8080, 0.0210)
LEFT_HEEL - 2D: (431, 582), 3D: (0.3367, 0.8086, 0.0446)
RIGHT_HEEL - 2D: (410, 581), 3D: (0.3208, 0.8072, 0.0395)
LEFT_FOOT_INDEX - 2D: (442, 611), 3D: (0.3460, 0.8492, 0.0405)
RIGHT_FOOT_INDEX - 2D: (423, 632), 3D: (0.3312, 0.8783, 0.0169)
LEFT_ANKLE - 2D: (466, 593), 3D: (0.3645, 0.8244, -0.0138)
RIGHT_ANKLE - 2D: (407, 591), 3D: (0.3184, 0.8217, 0.0165)
LEFT_HEEL - 2D: (468, 599), 3D: (0.3662, 0.8327, 0.0047)
RIGHT_HEEL - 2D: (417, 590), 3D: (0.3261, 0.8202, 0.0372)
LEFT_FOOT_INDEX - 2D: (467, 626), 3D: (0.3656, 0.8701, 0.0012)
RIGHT_FOOT_INDEX - 2D: (423, 641), 3D: (0.3311, 0.8911, 0.0335)
LEFT_ANKLE - 2D: (516, 613), 3D: (0.4033, 0.8521, -0.1567)
RIGHT_ANKLE - 2D: (388, 607), 3D: (0.3037, 0.8437, -0.1247)
LEFT_HEEL - 2D: (524, 624), 3D: (0.4098, 0.8671, -0.1518)
RIGHT_HEEL - 2D: (399, 617), 3D: (0.3119, 0.8571, -0.1191)
LEFT_FOOT_INDEX - 2D: (539, 651), 3D: (0.4218

KeyboardInterrupt: 

: 